![image info](https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/images/banner_1.png)

# Proyecto 1 - Predicción de precios de vehículos usados

En este proyecto podrán poner en práctica sus conocimientos sobre modelos predictivos basados en árboles y ensambles, y sobre la disponibilización de modelos. Para su desasrrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 1: Predicción de precios de vehículos usados".

**Entrega**: La entrega del proyecto deberán realizarla durante la semana 4. Sin embargo, es importante que avancen en la semana 3 en el modelado del problema y en parte del informe, tal y como se les indicó en la guía.

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 4, y subir el archivo de predicciones a la [competencia de Kaggle](https://www.kaggle.com/t/b8be43cf89c540bfaf3831f2c8506614).

## Datos para la predicción de precios de vehículos usados

En este proyecto se usará el conjunto de datos de Car Listings de Kaggle, donde cada observación representa el precio de un automóvil teniendo en cuenta distintas variables como: año, marca, modelo, entre otras. El objetivo es predecir el precio del automóvil. Para más detalles puede visitar el siguiente enlace: [datos](https://www.kaggle.com/jpayne/852k-used-car-listings).

## Ejemplo predicción conjunto de test para envío a Kaggle

En esta sección encontrarán el formato en el que deben guardar los resultados de la predicción para que puedan subirlos a la competencia en Kaggle.

In [13]:
import warnings
warnings.filterwarnings('ignore')

In [14]:
# Importación librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import joblib
from flask import Flask
from flask_restx import Api, Resource, fields, reqparse

In [15]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/datasets/dataTrain_carListings.zip')
dataTesting = pd.read_csv('https://raw.githubusercontent.com/davidzarruk/MIAD_ML_NLP_2023/main/datasets/dataTest_carListings.zip', index_col=0)

In [14]:
# Visualización datos de entrenamiento
dataTraining.head()

,Price,Year,Mileage,State,Make,Model
0,34995,2017,9913,FL,Jeep,Wrangler
1,37895,2015,20578,OH,Chevrolet,Tahoe4WD
2,18430,2012,83716,TX,BMW,X5AWD
3,24681,2014,28729,OH,Cadillac,SRXLuxury
4,26998,2013,64032,CO,Jeep,Wrangler


In [15]:
dataTraining.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400000 entries, 0 to 399999
Data columns (total 6 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Price    400000 non-null  int64 
 1   Year     400000 non-null  int64 
 2   Mileage  400000 non-null  int64 
 3   State    400000 non-null  object
 4   Make     400000 non-null  object
 5   Model    400000 non-null  object
dtypes: int64(3), object(3)
memory usage: 18.3+ MB


In [16]:
len(dataTraining['Model'].unique())

525

In [17]:
# Visualización datos de test
dataTesting.head()

,Year,Mileage,State,Make,Model
ID,,,,,
0,2014,31909,MD,Nissan,MuranoAWD
1,2017,5362,FL,Jeep,Wrangler
2,2014,50300,OH,Ford,FlexLimited
3,2004,132160,WA,BMW,5
4,2015,25226,MA,Jeep,Grand


In [18]:
dataTesting.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Year     100000 non-null  int64 
 1   Mileage  100000 non-null  int64 
 2   State    100000 non-null  object
 3   Make     100000 non-null  object
 4   Model    100000 non-null  object
dtypes: int64(2), object(3)
memory usage: 4.6+ MB


In [ ]:
correlation_matrix = dataTesting.corr()
dataTesting

In [16]:
label_encoders = {}
for column in ['State', 'Make', 'Model']:
    le = LabelEncoder()
    dataTraining[column] = le.fit_transform(dataTraining[column])
    label_encoders[column] = le
X = dataTraining[['Year', 'Mileage', 'State', 'Make', 'Model']]
y = dataTraining['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Ranfom Forest

# Correr Aquí

In [20]:
#Calibración de n_estimatos

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

rf = RandomForestRegressor()

# Definir el espacio de búsqueda de hiperparámetros
param_dist = {
    'n_estimators': randint(650, 850),  # Definir el rango de n_estimators según sea necesario
    'max_depth': [None, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100],  # Puedes ajustar estos valores según lo creas conveniente
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 20),
    'max_features': ['auto', 'sqrt'],  # Puedes incluir más opciones según lo necesites
    'bootstrap': [True, False]}

# Configurar la búsqueda aleatoria con validación cruzada
random_search = RandomizedSearchCV(estimator=rf, param_distributions=param_dist, n_iter=100, cv=3, scoring='neg_mean_squared_error', random_state=42)

# Realizar la búsqueda aleatoria
random_search.fit(X_train, y_train)

# Obtener el mejor modelo
best_rf_random = random_search.best_estimator_

# Obtener el mejor valor de n_estimators
best_n_estimators_random = best_rf_random.get_params()['n_estimators']
best_n_estimators_random

750

In [21]:
from sklearn.ensemble import RandomForestRegressor
rf_model = RandomForestRegressor(n_estimators=750, random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

In [22]:
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

In [23]:
rmse_test

3731.525115477954

In [24]:
for column in ['State', 'Make', 'Model']:
    dataTesting[column] = label_encoders[column].transform(dataTesting[column])

In [25]:
dataTesting

,Year,Mileage,State,Make,Model
ID,,,,,
0,2014,31909,20,27,305
1,2017,5362,9,17,489
2,2014,50300,35,10,211
3,2004,132160,47,2,27
4,2015,25226,19,17,248
...,...,...,...,...,...
99995,2015,82719,43,12,401
99996,2015,19711,44,2,27
99997,2016,48049,4,27,305


In [26]:
X_testing = dataTesting[['Year', 'Mileage', 'State', 'Make', 'Model']]
y_pred_testing = rf_model.predict(X_testing)
y_pred_testing

array([21847.00666667, 33964.92266667, 23814.076     , ...,
       23127.02      , 17255.84933333, 19022.55466667])

In [27]:
predicted_prices = pd.DataFrame({
    'Price': y_pred_testing
})
predicted_prices.reset_index(drop=True, inplace=True)

In [28]:
predicted_prices

,Price
0,21847.006667
1,33964.922667
2,23814.076000
3,10165.390667
4,32796.545333
...,...
99995,19665.800000
99996,39753.589333
99997,23127.020000
99998,17255.849333


In [29]:
predicted_prices.to_csv('test_submission.csv', index_label='ID')
predicted_prices.head()

,Price
0,21847.006667
1,33964.922667
2,23814.076000
3,10165.390667
4,32796.545333


In [31]:
joblib.dump(rf_model, 'model_deployment/car_price_reg_rf.pkl')
joblib.dump(label_encoders, 'model_deployment/label_encoder_rf.pkl')

FileNotFoundError: [Errno 2] No such file or directory: 'model_deployment/car_price_reg_rf.pkl'

#### Modelo Fabi XGBoot

In [32]:
xgb_model = xgb.XGBRegressor(objective='reg:squarederror')

param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 5, 7, 9],
    'gamma': [0, 0.1, 0.5, 1, 1.5, 2],
    'colsample_bytree': [0.3, 0.5, 0.7, 1.0],
    'subsample': [0.5, 0.75, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.01, 0.1, 1]
}

xgb_random = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=100,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=2,
    random_state=42
)

xgb_random.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.3, gamma=1.5, learning_rate=0.05, max_depth=9, n_estimators=400, reg_alpha=1, reg_lambda=0.01, subsample=0.5; total time=   2.1s
[CV] END colsample_bytree=0.3, gamma=1.5, learning_rate=0.05, max_depth=9, n_estimators=400, reg_alpha=1, reg_lambda=0.01, subsample=0.5; total time=   1.9s
[CV] END colsample_bytree=0.3, gamma=1.5, learning_rate=0.05, max_depth=9, n_estimators=400, reg_alpha=1, reg_lambda=0.01, subsample=0.5; total time=   1.8s
[CV] END colsample_bytree=0.3, gamma=1.5, learning_rate=0.05, max_depth=9, n_estimators=400, reg_alpha=1, reg_lambda=0.01, subsample=0.5; total time=   1.9s
[CV] END colsample_bytree=0.3, gamma=1.5, learning_rate=0.05, max_depth=9, n_estimators=400, reg_alpha=1, reg_lambda=0.01, subsample=0.5; total time=   1.8s
[CV] END colsample_bytree=0.3, gamma=0, learning_rate=0.05, max_depth=3, n_estimators=400, reg_alpha=1, reg_lambda=0.1, subsample=1.0; total time=   0.7

RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                          random_state=None, ...),
                   n_iter=100,
                   param_distributions={'colsample_bytree': [0.3, 0.5, 0.7,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.5, 1, 1.5, 2],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2,
                                                          0.3],
                                        'max_depth': [3, 5, 7, 9],
                                        'n_estimators': [100, 200, 300, 400,
                                                         500],
                                        'reg_alpha': [0, 0.01, 0.1, 1],
                                        'reg_lambda': [0.01, 0.1, 1],
                                        'subsample': [0.5, 0.75, 1.0]},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [33]:
best_params = xgb_random.best_params_
best_neg_mse = xgb_random.best_score_

best_rmse = np.sqrt(-best_neg_mse)

print("Mejores parámetros:", best_params)
print("Mejor RMSE:", best_rmse)

Mejores parámetros: {'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 1.5, 'colsample_bytree': 0.7}
Mejor RMSE: 3735.6385093719196


In [34]:
best_params = {'subsample': 1.0, 'reg_lambda': 1, 'reg_alpha': 1, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.2, 'gamma': 1.5, 'colsample_bytree': 0.7}
best_model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    gamma=best_params['gamma'],
    colsample_bytree=best_params['colsample_bytree'],
    subsample=best_params['subsample'],
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda']
)
best_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=1.5, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.2, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [35]:
y_pred = best_model.predict(X_test)

In [36]:
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE on Test Set: {rmse_test}")

RMSE on Test Set: 3742.6170018293683


In [38]:
X_testing = dataTesting[['Year', 'Mileage', 'State', 'Make', 'Model']]
y_pred_testing = best_model.predict(X_testing)
y_pred_testing

array([21113.232, 36850.25 , 15248.423, ..., 23295.938, 16490.596,
       18991.664], dtype=float32)

In [39]:
predicted_prices = pd.DataFrame({
    'Price': y_pred_testing
})
predicted_prices.reset_index(drop=True, inplace=True)

In [40]:
# Guardar predicciones en formato exigido en la competencia de kaggle
predicted_prices.to_csv('test_submission.csv', index_label='ID')
predicted_prices.head()

,Price
0,21113.232422
1,36850.250000
2,15248.422852
3,7312.602539
4,30860.330078


### Stacking

In [41]:
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LinearRegression

In [42]:
rf_predictions = cross_val_predict(rf_model, X_train, y_train, cv=5)
xgb_predictions = cross_val_predict(best_model, X_train, y_train, cv=5)

In [43]:
# Paso 4: Crear un conjunto de datos de entrenamiento para el meta-modelo
stacking_train = pd.DataFrame({'RandomForest': rf_predictions, 'XGBoost': xgb_predictions})

In [44]:
# Entrenar el meta-modelo (en este caso, un modelo de regresión lineal)
meta_model = LinearRegression()
meta_model.fit(stacking_train, y_train)

LinearRegression()

In [45]:
# Paso 5: Evaluar el rendimiento del modelo stacking en el conjunto de prueba
rf_test_predictions = rf_model.predict(X_test)
xgb_test_predictions = best_model.predict(X_test)
stacking_test = pd.DataFrame({'RandomForest': rf_test_predictions, 'XGBoost': xgb_test_predictions})
stacking_predictions = meta_model.predict(stacking_test)
stacking_rmse = mean_squared_error(y_test, stacking_predictions, squared=False)
print("RMSE of stacking model:", stacking_rmse)

RMSE of stacking model: 3570.483418370123


In [47]:
X_testing = dataTesting[['Year', 'Mileage', 'State', 'Make', 'Model']]
rf_testing_predictions = rf_model.predict(X_testing)
xgb_testing_predictions = best_model.predict(X_testing)
stacking_testing = pd.DataFrame({'RandomForest': rf_testing_predictions, 'XGBoost': xgb_testing_predictions})
stacking_predictions_testing = meta_model.predict(stacking_testing)

In [48]:
stacking_predictions_testing

array([21444.73013236, 35538.84055962, 19201.17904545, ...,
       23213.57835351, 16825.89074643, 18991.97265362])

In [49]:
predicted_prices_stacking = pd.DataFrame({
    'Price': stacking_predictions_testing
})
predicted_prices_stacking.reset_index(drop=True, inplace=True)

In [50]:
predicted_prices_stacking

,Price
0,21444.730132
1,35538.840560
2,19201.179045
3,8595.307486
4,31772.711672
...,...
99995,20467.765440
99996,38096.568155
99997,23213.578354
99998,16825.890746


In [51]:
predicted_prices_stacking.to_csv('test_submission.csv', index_label='ID')
predicted_prices_stacking.head()

,Price
0,21444.730132
1,35538.840560
2,19201.179045
3,8595.307486
4,31772.711672


# Stacking Polinomial

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import LinearRegression

In [ ]:
rf_predictions = cross_val_predict(rf_model, X_train, y_train, cv=5)
xgb_predictions = cross_val_predict(best_model, X_train, y_train, cv=5)

In [ ]:
# Paso 4: Crear un conjunto de datos de entrenamiento para el meta-modelo
stacking_train = pd.DataFrame({'RandomForest': rf_predictions, 'XGBoost': xgb_predictions})

In [69]:
from sklearn.preprocessing import PolynomialFeatures

poly_features = PolynomialFeatures(degree=2)  # Puedes ajustar el grado del polinomio según sea necesario
stacking_train_poly = poly_features.fit_transform(stacking_train)
stacking_train_poly

array([[1.00000000e+00, 2.52161333e+04, 2.35832246e+04, 6.35853380e+08,
        5.94677736e+08, 5.56168483e+08],
       [1.00000000e+00, 1.28203827e+04, 1.19020518e+04, 1.64362212e+08,
        1.52588858e+08, 1.41658836e+08],
       [1.00000000e+00, 1.37342520e+04, 1.39371104e+04, 1.88629678e+08,
        1.91415786e+08, 1.94243045e+08],
       ...,
       [1.00000000e+00, 1.32952827e+04, 1.25345967e+04, 1.76764541e+08,
        1.66651006e+08, 1.57116114e+08],
       [1.00000000e+00, 1.21120453e+04, 1.24291279e+04, 1.46701642e+08,
        1.50542161e+08, 1.54483221e+08],
       [1.00000000e+00, 4.52041853e+04, 3.91933594e+04, 2.04341837e+09,
        1.77170388e+09, 1.53611942e+09]])

In [70]:
meta_model_poly = LinearRegression()
meta_model_poly.fit(stacking_train_poly, y_train)

LinearRegression()

In [71]:
stacking_test_polinomial = np.column_stack((rf_test_predictions, xgb_test_predictions))
stacking_test_poly = poly_features.transform(stacking_test_polinomial)

In [72]:
stacking_predictions_poly = meta_model_poly.predict(stacking_test_poly)

# Paso 10: Calcular el error cuadrático medio (MSE)
stacking_rmse_poli = mean_squared_error(y_test, stacking_predictions_poly, squared=False)
print("RMSE of stacking model:", stacking_rmse_poli)

RMSE of stacking model: 3552.0738250874724


In [ ]:
for column in ['State', 'Make', 'Model']:
    dataTesting[column] = label_encoders[column].transform(dataTesting[column])

In [ ]:
# predecir con datos testing
X_testing = dataTesting[['Year', 'Mileage', 'State', 'Make', 'Model']]
rf_testing_predictions = rf_model.predict(X_testing)
xgb_testing_predictions = best_model.predict(X_testing)
stacking_testing = pd.DataFrame({'RandomForest': rf_testing_predictions, 'XGBoost': xgb_testing_predictions})
stacking_predictions_testing = meta_model.predict(stacking_testing)

In [73]:
stacking_testing_polinomial = pd.DataFrame({'RandomForest': rf_testing_predictions, 'XGBoost': xgb_testing_predictions})
stacking_testing_poly = poly_features.transform(stacking_testing_polinomial)

In [74]:
stacking_predictions_poly_testing = meta_model_poly.predict(stacking_testing_poly)

In [75]:
predicted_prices_stacking_poly = pd.DataFrame({
    'Price': stacking_predictions_poly_testing
})
predicted_prices_stacking_poly.reset_index(drop=True, inplace=True)
predicted_prices_stacking_poly

,Price
0,21435.438040
1,35753.157239
2,20393.662854
3,8991.049098
4,31674.643522
...,...
99995,20408.518887
99996,37864.490483
99997,23171.859269
99998,16848.631209


In [76]:
predicted_prices_stacking_poly.to_csv('test_submission_stacking_poly.csv', index_label='ID')
predicted_prices_stacking_poly.head()

,Price
0,21435.438040
1,35753.157239
2,20393.662854
3,8991.049098
4,31674.643522


### Ensamblaje por Voting

In [52]:
from sklearn.ensemble import VotingRegressor

ensemble_model = VotingRegressor([('RandomForest', rf_model), ('XGBoost', best_model)], weights=[0.5, 0.5])

# Entrenar el modelo ensamblado
ensemble_model.fit(X_train, y_train)

VotingRegressor(estimators=[('RandomForest',
                             RandomForestRegressor(n_estimators=750,
                                                   random_state=42)),
                            ('XGBoost',
                             XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=0.7, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=1.5, grow_...
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=0.2, max_bin=None,
                                          max_cat_threshold=None,
                                          max_cat_to_onehot=None,
                                          max_delta_step=None, max_depth=7,
                                          max_leaves=None,
                                          min_child_weight=None, missing=nan,
                                          monotone_constraints=None,
                                          multi_strategy=None, n_estimators=400,
                                          n_jobs=None, num_parallel_tree=None,
                                          random_state=None, ...))],
                weights=[0.5, 0.5])

In [53]:
# Hacer predicciones en el conjunto de prueba
ensemble_predictions = ensemble_model.predict(X_test)

# Calcular el error cuadrático medio (MSE)
ensemble_rmse = mean_squared_error(y_test, ensemble_predictions, squared=False)
print("RMSE of ensemble model:", ensemble_rmse)

RMSE of ensemble model: 3569.192173470399


In [54]:
# Hacer predicciones en el conjunto de prueba
ensemble_predictions_testing = ensemble_model.predict(X_testing)
ensemble_predictions_testing

array([21480.11954427, 35407.58633333, 19531.24942578, ...,
       23211.47875   , 16873.22251823, 19007.10936458])

In [56]:
predicted_prices_voting = pd.DataFrame({
    'Price': ensemble_predictions_testing
})
predicted_prices_voting.reset_index(drop=True, inplace=True)
predicted_prices_voting

,Price
0,21480.119544
1,35407.586333
2,19531.249426
3,8738.996603
4,31828.437706
...,...
99995,20422.393164
99996,38182.132557
99997,23211.478750
99998,16873.222518


In [57]:
predicted_prices_voting.to_csv('test_submission_voting.csv', index_label='ID')
predicted_prices_voting.head()

,Price
0,21480.119544
1,35407.586333
2,19531.249426
3,8738.996603
4,31828.437706


In [14]:
joblib.dump(best_model, 'model_deployment/car_price_reg.pkl')
joblib.dump(label_encoders, 'model_deployment/label_encoders.pkl')

['model_deployment/label_encoders.pkl']

In [15]:
best_model = joblib.load('model_deployment/car_price_reg.pkl')
label_encoders = joblib.load('model_deployment/label_encoders.pkl')

app = Flask(__name__)
api = Api(app, version='1.0', title='Model API',
          description='A simple API that use model to make predictions')

ns = api.namespace('predict', description='Model Prediction')

model = api.model('PredictionData', {
    'Year': fields.Integer(required=True, description='Year of the vehicle'),
    'Mileage': fields.Integer(required=True, description='Mileage of the vehicle'),
    'State': fields.String(required=True, description='State where the vehicle is registered'),
    'Make': fields.String(required=True, description='Make of the vehicle'),
    'Model': fields.String(required=True, description='Model of the vehicle'),
})

parser = reqparse.RequestParser()
parser.add_argument('Year', type=int, required=True, help='Year of the vehicle')
parser.add_argument('Mileage', type=int, required=True, help='Mileage of the vehicle')
parser.add_argument('State', type=str, required=True, help='State where the vehicle is registered')
parser.add_argument('Make', type=str, required=True, help='Make of the vehicle')
parser.add_argument('Model', type=str, required=True, help='Model of the vehicle')

@ns.route('/')
class CarPriceApi(Resource):
    @api.expect(model)
    @api.response(200, 'Success')
    def post(self):
        args = parser.parse_args()
        input_data = pd.DataFrame([args])

        # Aplicar label encoding
        for column in ['State', 'Make', 'Model']:
            input_data[column] = label_encoders[column].transform(input_data[column])

        # Predecir con el modelo
        prediction = best_model.predict(input_data)

        # Devolver el resultado
        return {
            "result": float(prediction[0])
        }, 200

if __name__ == '__main__':
    app.run(debug=True, use_reloader=False, host='0.0.0.0', port=5000)

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: on


 * Running on all addresses.
 * Running on http://192.168.1.105:5000/ (Press CTRL+C to quit)
192.168.1.105 - - [21/Apr/2024 18:12:32] "GET / HTTP/1.1" 200 -
192.168.1.105 - - [21/Apr/2024 18:12:32] "GET /swagger.json HTTP/1.1" 200 -
192.168.1.105 - - [21/Apr/2024 18:12:32] "GET /swaggerui/swagger-ui.css HTTP/1.1" 200 -
192.168.1.105 - - [21/Apr/2024 18:12:32] "GET /swaggerui/favicon-16x16.png HTTP/1.1" 200 -
192.168.1.105 - - [21/Apr/2024 18:12:48] "POST /predict/ HTTP/1.1" 200 -


In [7]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
xgb_model = xgb.XGBRegressor(objective='reg:squarederror')

param_dist = {
    'n_estimators': randint(50, 1000),
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': randint(0, 10),
    'gamma': [0, 0.1, 0.5, 1, 1.5, 2],
    'colsample_bytree': [0.3, 0.5, 0.7, 1.0],
    'subsample': [0.5, 0.75, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0.01, 0.1, 1]
}

xgb_random_2 = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=100,
    scoring='neg_mean_squared_error',
    cv=5,
    verbose=2,
    random_state=42
)

xgb_random_2.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.3, max_depth=7, n_estimators=750, reg_alpha=0, reg_lambda=1, subsample=0.75; total time=   3.3s
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.3, max_depth=7, n_estimators=750, reg_alpha=0, reg_lambda=1, subsample=0.75; total time=   3.1s
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.3, max_depth=7, n_estimators=750, reg_alpha=0, reg_lambda=1, subsample=0.75; total time=   3.3s
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.3, max_depth=7, n_estimators=750, reg_alpha=0, reg_lambda=1, subsample=0.75; total time=   3.2s
[CV] END colsample_bytree=0.7, gamma=1, learning_rate=0.3, max_depth=7, n_estimators=750, reg_alpha=0, reg_lambda=1, subsample=0.75; total time=   3.3s
[CV] END colsample_bytree=0.7, gamma=0.5, learning_rate=0.1, max_depth=7, n_estimators=422, reg_alpha=1, reg_lambda=1, subsample=0.75; total time=   1.9s
[CV] END colsample_bytr

RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          gamma=None, grow_policy=None,
                                          importance_type=None,
                                          interaction_constraints=None,
                                          learning_rate=...
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2,
                                                          0.3],
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x16ab0a640>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x16ab90070>,
                                        'reg_alpha': [0, 0.01, 0.1, 1],
                                        'reg_lambda': [0.01, 0.1, 1],
                                        'subsample': [0.5, 0.75, 1.0]},
                   random_state=42, scoring='neg_mean_squared_error',
                   verbose=2)

In [8]:
best_params_2 = xgb_random_2.best_params_
best_neg_mse_2 = xgb_random_2.best_score_

best_rmse_2 = np.sqrt(-best_neg_mse_2)

print("Mejores parámetros:", best_params_2)
print("Mejor RMSE:", best_rmse_2)

Mejores parámetros: {'colsample_bytree': 0.7, 'gamma': 1.5, 'learning_rate': 0.05, 'max_depth': 8, 'n_estimators': 813, 'reg_alpha': 1, 'reg_lambda': 0.01, 'subsample': 0.5}
Mejor RMSE: 3742.5919444561573


In [10]:
best_params_2 = {'subsample': 0.5, 'reg_lambda': 0.01, 'reg_alpha': 1, 'n_estimators': 813, 'max_depth': 8, 'learning_rate': 0.05, 'gamma': 1.5, 'colsample_bytree': 0.7}
best_model_2 = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=best_params_2['n_estimators'],
    learning_rate=best_params_2['learning_rate'],
    max_depth=best_params_2['max_depth'],
    gamma=best_params_2['gamma'],
    colsample_bytree=best_params_2['colsample_bytree'],
    subsample=best_params_2['subsample'],
    reg_alpha=best_params_2['reg_alpha'],
    reg_lambda=best_params_2['reg_lambda']
)
best_model_2.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=1.5, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=8, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=813, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [11]:
y_pred_2 = best_model_2.predict(X_test)

In [12]:
rmse_test_2 = np.sqrt(mean_squared_error(y_test, y_pred_2))
print(f"RMSE on Test Set: {rmse_test_2}")

RMSE on Test Set: 3753.9010365353024
